# Notebook 3: Data Efficiency

This notebook evaluates **Zonal** data-efficiency ablations across:
- `Edge_10`
- `Edge_25`
- `Edge_50`
- `Edge_75`
- `Edge` (100%)

for the two families:
- `ArGEnT_self_att_noSDF`
- `PointNetMLPJoint_FP`

All outputs are labelled **validation-split evaluation**. The same deterministic 20% geometry holdout (`seed=42`) is enforced for every fraction/model combination.


In [ ]:
from __future__ import annotations
import ast, hashlib, importlib.util, itertools, json, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
try:
    import torch
    import h5py
except ImportError as exc:
    raise RuntimeError('Install torch and h5py in the selected notebook kernel before executing this comparison.') from exc
CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
if not (REPO_ROOT / 'Uniform').exists(): raise RuntimeError(f'Repository root not found from {CURRENT_DIR}')
COMPARISON_DIR = REPO_ROOT / 'Comparison'
RESULTS_DIR = COMPARISON_DIR / 'results' / '03_data_efficiency'
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(COMPARISON_DIR))
import eval_helpers as eh
SPLIT_SEED, EVAL_FRACTION = 42, 0.20
REGIME = 'Zonal'
ABLATIONS = ['Edge_10', 'Edge_25', 'Edge_50', 'Edge_75', 'Edge']
ABLATION_TO_FRACTION = {'Edge_10': 0.10, 'Edge_25': 0.25, 'Edge_50': 0.50, 'Edge_75': 0.75, 'Edge': 1.00}
FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint_FP']
DATASET_PATH = REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_zonal.h5'
COMMIT = __import__('subprocess').check_output(['git','rev-parse','HEAD'], cwd=REPO_ROOT, text=True).strip()
display(Markdown(
    f'**Inspected commit:** `{COMMIT}`\n\n'
    f'**Results directory:** `{RESULTS_DIR}`\n\n'
    f'**Evaluation label:** `validation-split evaluation`'
))


## Discovery, architecture consistency, and split metadata


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

def decode(value):
    return value.decode() if isinstance(value, bytes) else value

def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value

def compact_hash(obj):
    payload = json.dumps(json_ready(obj), sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()[:16]

def parse_script_metadata(script_path: Path) -> dict:
    meta = {'path': str(script_path)}
    tree = ast.parse(script_path.read_text(encoding='utf-8'))
    wanted = {
        'INPUT_COLS', 'EXTRA_FEAT_COLS', 'HEAD_FEAT_COLS', 'TARGET_NAMES',
        'EXPECTED_REPR', 'H5_FILENAME', 'Perc_training_data', 'train_data_percent'
    }
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in wanted:
                    try:
                        meta[target.id] = ast.literal_eval(node.value)
                    except Exception:
                        pass
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id in wanted:
            try:
                meta[node.target.id] = ast.literal_eval(node.value)
            except Exception:
                pass
        elif isinstance(node, ast.Call) and getattr(node.func, 'id', None) == 'train_test_split':
            for kw in node.keywords:
                if kw.arg in {'test_size', 'random_state'}:
                    try:
                        meta[kw.arg] = ast.literal_eval(kw.value)
                    except Exception:
                        pass
    return meta

def discover_checkpoints():
    rows = []
    for ablation in ABLATIONS:
        for family in FAMILIES:
            folder = REPO_ROOT / REGIME / ablation / family
            checkpoints = sorted((folder / 'Trained_models').glob('*.pt'))
            scripts = sorted(folder.glob('Training_script*.py'))
            script_meta = parse_script_metadata(scripts[0]) if scripts else {}
            if not checkpoints:
                rows.append({
                    'regime': REGIME,
                    'ablation': ablation,
                    'model_family': family,
                    'status': 'missing checkpoint',
                    'checkpoint_path': None,
                })
                continue
            for path in checkpoints:
                status = 'discovered'
                metadata = {}
                try:
                    payload = torch.load(path, map_location='cpu', weights_only=False)
                    required = ['arch', 'model_state', 'coord_center', 'coord_half_range', 'target_mean', 'target_std']
                    missing = [k for k in required if k not in payload]
                    if missing:
                        status = 'incompatible: missing ' + ', '.join(missing)
                    metadata = {
                        'arch': json_ready(payload.get('arch')),
                        'arch_hash': compact_hash(payload.get('arch')) if payload.get('arch') is not None else None,
                        'model_name': payload.get('model_name'),
                        'extra_feat_cols': json_ready(payload.get('extra_feat_cols')),
                        'target_names': json_ready(payload.get('target_names')),
                        'script_meta': json_ready(script_meta),
                        'script_fraction': script_meta.get('Perc_training_data', script_meta.get('train_data_percent')),
                        'script_test_size': script_meta.get('test_size'),
                        'script_split_seed': script_meta.get('random_state'),
                    }
                    if script_meta.get('EXPECTED_REPR') not in (None, 'edge'):
                        status = f"incompatible: training script expects representation {script_meta.get('EXPECTED_REPR')!r}"
                    if script_meta.get('H5_FILENAME') not in (None, DATASET_PATH.name):
                        status = f"incompatible: training script points to {script_meta.get('H5_FILENAME')!r}"
                except Exception as exc:
                    status = 'incompatible: ' + type(exc).__name__ + ': ' + str(exc)
                rows.append({
                    'regime': REGIME,
                    'ablation': ablation,
                    'training_fraction': ABLATION_TO_FRACTION[ablation],
                    'model_family': family,
                    'status': status,
                    'checkpoint_path': str(path),
                    'file_size_bytes': path.stat().st_size,
                    'sha256': sha256(path),
                    **metadata,
                })
    return pd.DataFrame(rows)

checkpoint_report = discover_checkpoints()
eh.save_table(checkpoint_report, RESULTS_DIR, 'checkpoint_integrity')

def first_nonnull(values):
    for value in values:
        if value is not None and not (isinstance(value, float) and np.isnan(value)):
            return value
    return None

arch_consistency_rows = []
for family, sub in checkpoint_report[checkpoint_report.status == 'discovered'].groupby('model_family'):
    unique_arch = sorted(set(sub.arch_hash.dropna()))
    unique_extra = sorted(set(json.dumps(v) for v in sub.extra_feat_cols.dropna())) if 'extra_feat_cols' in sub else []
    arch_consistency_rows.append({
        'model_family': family,
        'n_checkpoints': len(sub),
        'unique_arch_hashes': unique_arch,
        'unique_extra_feat_cols': unique_extra,
        'status': 'consistent' if len(unique_arch) <= 1 and len(unique_extra) <= 1 else 'discrepant',
    })
architecture_consistency = pd.DataFrame(arch_consistency_rows)
eh.save_table(architecture_consistency, RESULTS_DIR, 'architecture_consistency')

split_consistency_rows = []
for family, sub in checkpoint_report[checkpoint_report.status == 'discovered'].groupby('model_family'):
    seeds = sorted(set(x for x in sub.script_split_seed.dropna()))
    test_sizes = sorted(set(x for x in sub.script_test_size.dropna()))
    fractions = sorted(set(round(float(x), 6) for x in sub.script_fraction.dropna()))
    split_consistency_rows.append({
        'model_family': family,
        'split_seeds_from_scripts': seeds,
        'split_test_sizes_from_scripts': test_sizes,
        'fractions_from_scripts': fractions,
        'expected_holdout_fraction': EVAL_FRACTION,
        'status': 'consistent' if seeds == [SPLIT_SEED] and test_sizes == [EVAL_FRACTION] else 'discrepant_or_unrecorded',
    })
split_consistency = pd.DataFrame(split_consistency_rows)
eh.save_table(split_consistency, RESULTS_DIR, 'split_consistency')
display(checkpoint_report[['ablation', 'model_family', 'status', 'checkpoint_path', 'arch_hash', 'script_fraction', 'script_split_seed', 'script_test_size']])
display(architecture_consistency)
display(split_consistency)


## Fixed evaluation split and training-count estimates


In [ ]:
def load_samples(path):
    samples = []
    if not path.exists():
        raise FileNotFoundError(f'Dataset not found: {path}')
    with h5py.File(path, 'r') as h5:
        representation = decode(h5.attrs.get('representation', ''))
        if representation != 'edge': raise ValueError(f'{path.name}: expected representation edge, got {representation!r}')
        for key in sorted(h5['samples'].keys()):
            g = h5['samples'][key]
            def arr(name, default=None): return np.asarray(g[name]) if name in g else default
            coords = arr('node_coords_mm'); stress = arr('stress_max_vm'); life = arr('life_raw')
            if coords is None or stress is None or life is None: raise ValueError(f'{path.name}/{key}: missing required target fields')
            sample_id = decode(g.attrs.get('sample_id', key))
            attrs = {str(k): decode(v) for k, v in g.attrs.items()}
            samples.append({
                'sample_key': key,
                'sample_id': str(sample_id),
                'attrs': attrs,
                'coords': coords.astype('float32'),
                'stress': stress.astype('float32').reshape(-1),
                'loglife': np.log10(np.clip(life.astype('float64').reshape(-1), 1e-30, None)).astype('float32'),
                'zone_id': arr('zone_id', np.full(len(coords), -1)).reshape(-1),
                'subzone_id': arr('subzone_id', np.full(len(coords), np.nan)).reshape(-1),
                'arc_length_mm': arr('arc_length_mm', np.arange(len(coords), dtype='float32')).reshape(-1),
                'node_features': arr('node_features', np.empty((len(coords), 0), dtype='float32')),
            })
    return samples

def split_samples(samples):
    rng = np.random.default_rng(SPLIT_SEED); order = rng.permutation(len(samples)); n_eval = max(1, int(round(len(samples) * EVAL_FRACTION)))
    eval_pos = np.sort(order[:n_eval]).tolist(); train_pos = np.sort(order[n_eval:]).tolist()
    return train_pos, eval_pos

all_samples = load_samples(DATASET_PATH)
train_pos, eval_pos = split_samples(all_samples)
full_train_count = len(train_pos)
training_count_rows = []
for _, row in checkpoint_report.iterrows():
    if row.status != 'discovered':
        continue
    fraction = row.training_fraction
    training_count_rows.append({
        'ablation': row.ablation,
        'training_fraction': fraction,
        'model_family': row.model_family,
        'training_geometries': int(len(train_pos) * fraction),
        'count_source': 'estimated_from_notebook_split',
        'evaluation_geometries': len(eval_pos),
        'evaluation_label': 'validation-split evaluation',
    })
training_counts = pd.DataFrame(training_count_rows).sort_values(['training_fraction', 'model_family']).reset_index(drop=True)
eh.save_table(training_counts, RESULTS_DIR, 'training_geometry_counts')
with open(RESULTS_DIR / 'evaluation_split_provenance.json', 'w', encoding='utf-8') as stream:
    json.dump({
        'regime': REGIME,
        'dataset_path': str(DATASET_PATH),
        'total_geometry_count': len(all_samples),
        'training_geometry_count_full': len(train_pos),
        'evaluation_geometry_count': len(eval_pos),
        'training_sample_ids_full': [all_samples[i]['sample_id'] for i in train_pos],
        'evaluation_sample_ids': [all_samples[i]['sample_id'] for i in eval_pos],
        'split_seed': SPLIT_SEED,
        'split_fraction': EVAL_FRACTION,
        'evaluation_label': 'validation-split evaluation',
        'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'notebook_commit_sha': COMMIT,
    }, stream, indent=2)
display(training_counts)


## Reconstruction and inference on the shared holdout geometries


In [ ]:
def import_local(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def reconstruct(row):
    folder = Path(row['checkpoint_path']).parent.parent; family = row['model_family']
    ckpt = torch.load(row['checkpoint_path'], map_location='cpu', weights_only=False); arch = dict(ckpt['arch'])
    pn = import_local(folder / 'pn_models.py', f"pn_{folder.parent.name}_{folder.name}_{family}_{row.ablation}")
    sys.modules['pn_models'] = pn
    if family in ('PointNetMLPJoint_FP', 'PointNetMLPJoint_FP_headfeat'):
        if not hasattr(pn, 'build_fp_model_from_arch'): raise RuntimeError('FP checkpoint requires build_fp_model_from_arch')
        model = pn.build_fp_model_from_arch(arch)
    elif family in ('PointNetMLPJoint', 'PointNetMLPJoint_weighted', 'PointNetMLPJoint_headfeat'):
        model = pn.build_model_from_arch(arch)
    else:
        bench = import_local(folder / 'benchmarks.py', f'bench_{family}_{row.ablation}')
        if not hasattr(bench, 'ArGEnTDeepONet'): raise RuntimeError('No ArGEnTDeepONet found in own benchmarks.py')
        defaults = {'hidden_dim': 128, 'num_heads': 4, 'num_layers': 2, 'output_dim': 128, 'out_channels': 1, 'attention_type': 'self', 'use_sdf': False, 'in_ch_geom': 2}
        defaults.update({k: v for k, v in arch.items() if k in defaults})
        if 'out_channels' not in arch and 'bias' in ckpt['model_state']: defaults['out_channels'] = int(ckpt['model_state']['bias'].shape[0])
        model = bench.ArGEnTDeepONet(**defaults)
    model.load_state_dict(ckpt['model_state'], strict=True)
    model.eval()
    return model, ckpt

def normalize_query(coords, ckpt):
    center = np.asarray(ckpt['coord_center'], dtype='float32')
    half = np.maximum(np.asarray(ckpt['coord_half_range'], dtype='float32'), 1e-8)
    return ((coords.astype('float32') - center) / half).astype('float32')

def feature_matrix(sample, ckpt):
    cols = ckpt.get('extra_feat_cols', []) or []
    if not cols:
        return normalize_query(sample['coords'], ckpt), np.empty((len(sample['coords']), 0), dtype='float32')
    available = sample['node_features'].astype('float32')
    if available.shape[1] < len(cols):
        raise ValueError(f'missing required extra-feature columns: {cols}')
    extra = available[:, :len(cols)]
    stats = ckpt.get('extra_feat_stats')
    if stats is None:
        raise ValueError('missing extra-feature normalization statistics')
    if isinstance(stats, dict) and 'mean' in stats and 'std' in stats:
        mean = np.asarray(stats['mean'], dtype='float32'); std = np.asarray(stats['std'], dtype='float32')
    elif isinstance(stats, dict):
        mean = np.asarray([stats[c]['mean'] if c in stats else stats[str(c)]['mean'] for c in cols], dtype='float32')
        std = np.asarray([stats[c]['std'] if c in stats else stats[str(c)]['std'] for c in cols], dtype='float32')
    else:
        mean = np.asarray(stats[0], dtype='float32'); std = np.asarray(stats[1], dtype='float32')
    coords = normalize_query(sample['coords'], ckpt)
    extra = (extra - mean) / np.maximum(std, 1e-8)
    return coords, extra.astype('float32')

def decode_prediction(out, ckpt):
    out = out.detach().cpu().numpy()
    if out.ndim != 3 or out.shape[0] != 1:
        raise ValueError(f'prediction shape unexpected: {out.shape}')
    mean = np.asarray(ckpt['target_mean'], dtype='float32')
    std = np.asarray(ckpt['target_std'], dtype='float32')
    out = out * std + mean
    if out.shape[2] == 2: return out[0, :, 0], out[0, :, 1]
    if out.shape[2] == 1: return np.zeros(out.shape[1], dtype='float32'), out[0, :, 0]
    raise ValueError(f'unexpected output channels: {out.shape[2]}')

def predict(model, sample, ckpt, family):
    coords, extra = feature_matrix(sample, ckpt)
    x = torch.from_numpy(coords[None]); q = x.clone(); extra_t = torch.from_numpy(extra[None])
    with torch.no_grad():
        try:
            out = model(x, q)
        except TypeError:
            try:
                out = model(x, q, geom_feats=extra_t)
            except TypeError:
                out = model(torch.cat([x, extra_t], dim=-1), q)
    return decode_prediction(out, ckpt)

node_frames, load_errors, coverage_rows = [], [], []
for _, row in checkpoint_report.iterrows():
    if row.status != 'discovered':
        continue
    try:
        model, ckpt = reconstruct(row)
        predicted_ids = []
        training_config_id = f"{row.ablation}:{row.model_family}:{Path(row['checkpoint_path']).stem}"
        for i in eval_pos:
            s = all_samples[i]
            pred_stress, pred_loglife = predict(model, s, ckpt, row.model_family)
            predicted_ids.append(s['sample_id'])
            base = pd.DataFrame({
                'regime': REGIME,
                'ablation': row.ablation,
                'training_fraction': row.training_fraction,
                'model_family': row.model_family,
                'checkpoint_path': row.checkpoint_path,
                'training_config_id': training_config_id,
                'sample_key': s['sample_key'],
                'sample_id': s['sample_id'],
                'node_idx': np.arange(len(s['coords'])),
                'x_mm': s['coords'][:, 0],
                'r_mm': s['coords'][:, 1],
                'zone_id': s['zone_id'],
                'subzone_id': s['subzone_id'],
                'arc_length_mm': s['arc_length_mm'],
                'true_stress': s['stress'],
                'pred_stress': pred_stress,
                'true_loglife': s['loglife'],
                'pred_loglife': pred_loglife,
            })
            base['zone_name'] = base.zone_id.map(eh.ZONE_ID_TO_NAME)
            base['subzone_name'] = base.subzone_id.map(eh.SUBZONE_ID_TO_NAME)
            node_frames.append(base)
        coverage_rows.append({
            'ablation': row.ablation,
            'training_fraction': row.training_fraction,
            'model_family': row.model_family,
            'checkpoint_path': row.checkpoint_path,
            'training_config_id': training_config_id,
            'sample_ids': predicted_ids,
        })
    except Exception as exc:
        load_errors.append({'ablation': row.ablation, 'model_family': row.model_family, 'status': 'load/inference failed: ' + repr(exc)})

nodes = pd.concat(node_frames, ignore_index=True) if node_frames else pd.DataFrame()
pd.DataFrame(load_errors).to_json(RESULTS_DIR / 'inference_errors.json', orient='records', indent=2)
coverage = pd.DataFrame(coverage_rows)
if coverage.empty:
    raise RuntimeError('No model/fraction combinations completed inference.')
shared_ids = sorted(set.intersection(*[set(ids) for ids in coverage['sample_ids']]))
if not shared_ids:
    raise RuntimeError('No shared evaluation geometries across all successful model/fraction combinations.')
coverage_summary = coverage.copy()
coverage_summary['successful_eval_geometries'] = coverage_summary['sample_ids'].map(len)
coverage_summary['shared_eval_geometries'] = len(shared_ids)
coverage_summary['dropped_to_enforce_fairness'] = coverage_summary['successful_eval_geometries'] - len(shared_ids)
coverage_summary['evaluation_label'] = 'validation-split evaluation'
eh.save_table(coverage_summary.drop(columns=['sample_ids']), RESULTS_DIR, 'evaluation_geometry_coverage')
nodes = nodes[nodes.sample_id.isin(shared_ids)].copy()
display(pd.DataFrame(load_errors) if load_errors else coverage_summary.drop(columns=['sample_ids']))



## Per-fraction metrics and trend figures

This cell computes pooled metrics plus full life-band and grouped-region diagnostics for every training fraction.


In [ ]:
pooled = eh.pooled_metrics_from_nodes(nodes)
life_bands = eh.life_bin_metrics_by_groups(
    nodes,
    group_cols=['regime', 'ablation', 'training_fraction', 'model_family', 'checkpoint_path', 'training_config_id'],
    include_full_set=True,
)
grouped_regions = eh.grouped_region_metrics_from_nodes(nodes)
geom = eh.geometry_level_metrics(nodes)

if life_bands.empty:
    raise RuntimeError('No life-bin rows were generated from notebook inference output.')

life_bands['training_fraction'] = pd.to_numeric(life_bands['training_fraction'], errors='coerce')
life_bands = life_bands[life_bands['training_fraction'].notna()].copy()
life_bands['training_fraction_pct'] = life_bands['training_fraction'] * 100.0
life_bands['training_geometries'] = (life_bands['training_fraction'] * full_train_count).round().astype(int)
life_bands['evaluation_label'] = 'validation-split evaluation'

if life_bands.empty:
    raise RuntimeError('Life-bin table is empty after coercing training fractions to numeric values.')

summary_rows = []
full_set_rows = life_bands[life_bands['life_bin'] == eh.FULL_TEST_SET_LABEL].copy()
for ablation in ABLATIONS:
    frac = ABLATION_TO_FRACTION[ablation]
    for family in FAMILIES:
        row = {
            'ablation': ablation,
            'training_fraction': frac,
            'training_fraction_pct': frac * 100.0,
            'model_family': family,
            'training_geometries': int(full_train_count * frac),
        }
        for target in ('LogLife', 'Stress'):
            sub = pooled[(pooled.ablation == ablation) & (pooled.model_family == family) & (pooled.target == target)]
            if not sub.empty:
                row[f'{target}_MAE'] = float(sub.iloc[0]['MAE'])
                row[f'{target}_RMSE'] = float(sub.iloc[0]['RMSE'])
                row[f'{target}_R2'] = float(sub.iloc[0]['R2 (log)'])
        fs = full_set_rows[
            (full_set_rows['ablation'] == ablation) &
            (full_set_rows['model_family'] == family)
        ]
        if not fs.empty:
            row['n_samples_full_test'] = int(fs['n_samples'].median())
            row['LogLife_MAE'] = float(fs['mae_loglife'].mean())
            row['LogLife_RMSE'] = float(fs['rmse_loglife'].mean())
            row['run_count'] = int(fs[['checkpoint_path', 'training_config_id']].drop_duplicates().shape[0])
        sub = grouped_regions[(grouped_regions.ablation == ablation) & (grouped_regions.model_family == family)
                              & (grouped_regions.grouped_region == 'Critical lower transition')]
        if not sub.empty and sub.iloc[0].get('status') == 'ok':
            row['critical_lower_transition_MAE'] = float(sub.iloc[0]['LogLife_MAE'])
        gsub = geom[(geom.ablation == ablation) & (geom.model_family == family)]
        if not gsub.empty:
            row['whole_geometry_mean_LogLife_MAE'] = float(gsub['whole_geometry_loglife_mae'].mean())
            row['abs_min_loglife_error_mean'] = float(gsub['abs_min_loglife_error_decades'].mean())
        summary_rows.append(row)

summary_by_fraction = pd.DataFrame(summary_rows).sort_values(['training_fraction', 'model_family']).reset_index(drop=True)
eh.save_table(summary_by_fraction, RESULTS_DIR, 'summary_by_fraction')
eh.save_table(life_bands, RESULTS_DIR, 'data_efficiency_by_life_bin')
eh.save_table(life_bands, RESULTS_DIR, 'life_band_metrics')
eh.save_table(grouped_regions, RESULTS_DIR, 'grouped_region_metrics')

run_metadata = {
    'commit_sha': COMMIT,
    'evaluation_label': 'validation-split evaluation',
    'results_dir': str(RESULTS_DIR),
    'families': FAMILIES,
    'ablations': ABLATIONS,
    'ablation_to_fraction': ABLATION_TO_FRACTION,
    'quantitative_only_from_production_hdf5': True,
    'qualitative_examples_use_example_hdf5_only': True,
}
eh.save_json(run_metadata, RESULTS_DIR, 'run_metadata')

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
family_labels = {'ArGEnT_self_att_noSDF': 'ArGEnT', 'PointNetMLPJoint_FP': 'PointNet+FP'}
colors = {'ArGEnT_self_att_noSDF': '#1f77b4', 'PointNetMLPJoint_FP': '#d62728'}


def save_fig(fig, name):
    fig.savefig(FIGURES_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    fig.savefig(FIGURES_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.close(fig)


def print_data_efficiency_diagnostics(metrics_df: pd.DataFrame):
    model_variants = sorted(metrics_df['model_family'].dropna().astype(str).unique().tolist())
    fractions = sorted(pd.to_numeric(metrics_df['training_fraction'], errors='coerce').dropna().unique().tolist())
    life_bins = sorted(metrics_df['life_bin'].dropna().astype(str).unique().tolist())
    counts = metrics_df.groupby(['life_bin', 'training_fraction'], as_index=False).size().rename(columns={'size': 'rows'})
    non_null_mae = int(metrics_df['mae_loglife'].notna().sum()) if 'mae_loglife' in metrics_df.columns else 0
    non_null_rmse = int(metrics_df['rmse_loglife'].notna().sum()) if 'rmse_loglife' in metrics_df.columns else 0
    display(Markdown(
        "\n".join([
            "**Data-efficiency diagnostics**",
            f"- model variants: `{model_variants}`",
            f"- training fractions: `{fractions}`",
            f"- life bins: `{life_bins}`",
            f"- non-null MAE rows: `{non_null_mae}`",
            f"- non-null RMSE rows: `{non_null_rmse}`",
        ])
    ))
    display(counts)


print_data_efficiency_diagnostics(life_bands)

per_bin_df, full_set_df = eh.split_life_bin_metrics(life_bands, bin_col='life_bin')
eh.assert_non_empty_plot_df(
    per_bin_df,
    intended_plot='Notebook 3 per-life-bin data-efficiency plot',
    reference_df=life_bands,
    metric_cols=('mae_loglife', 'rmse_loglife'),
)
eh.assert_non_empty_plot_df(
    full_set_df,
    intended_plot='Notebook 3 full-test-set data-efficiency summary plot',
    reference_df=life_bands,
    metric_cols=('mae_loglife', 'rmse_loglife'),
)



def _plot_runs_and_summary(ax, panel_df: pd.DataFrame, metric_col: str):
    metric_df = panel_df.dropna(subset=['training_fraction_pct', metric_col]).copy()
    if metric_df.empty:
        return False
    for family in FAMILIES:
        fam = metric_df[metric_df['model_family'] == family].copy()
        if fam.empty:
            continue
        fam = fam.sort_values(['training_fraction_pct', 'checkpoint_path'])
        ax.scatter(
            fam['training_fraction_pct'],
            fam[metric_col],
            alpha=0.45,
            s=18,
            color=colors.get(family, None),
            label=f"{family_labels.get(family, family)} runs",
        )
        stats = fam.groupby('training_fraction_pct', as_index=False)[metric_col].agg(['mean', 'std', 'count']).reset_index()
        stats = stats.sort_values('training_fraction_pct')
        ax.plot(
            stats['training_fraction_pct'],
            stats['mean'],
            marker='o',
            lw=2,
            color=colors.get(family, None),
            label=f"{family_labels.get(family, family)} mean",
        )
        if (stats['count'] > 1).any() and stats['std'].notna().any():
            spread = stats['std'].fillna(0.0)
            ax.fill_between(
                stats['training_fraction_pct'],
                stats['mean'] - spread,
                stats['mean'] + spread,
                alpha=0.12,
                color=colors.get(family, None),
            )
    return True



def faceted_metric_plot(metric_col: str, metric_label: str, filename: str):
    bins = [b for b in eh.PHYSICAL_LIFE_BIN_ORDER if b in per_bin_df['life_bin'].astype(str).unique()]
    eh.assert_non_empty_plot_df(
        per_bin_df[per_bin_df['life_bin'].astype(str).isin(bins)],
        intended_plot=f'{metric_label} vs training fraction by life bin',
        reference_df=life_bands,
        metric_cols=(metric_col,),
    )
    fig, axes = plt.subplots(1, len(bins), figsize=(5.2 * len(bins), 4.2), sharey=False)
    if len(bins) == 1:
        axes = [axes]
    for ax, life_bin in zip(axes, bins):
        panel = per_bin_df[per_bin_df['life_bin'].astype(str) == life_bin].copy()
        if panel.empty or panel[metric_col].isna().all():
            ax.text(0.5, 0.5, f'No valid data for {life_bin}', transform=ax.transAxes, ha='center', va='center')
            ax.set_axis_off()
            continue
        plotted = _plot_runs_and_summary(ax, panel, metric_col)
        if not plotted:
            ax.text(0.5, 0.5, f'No valid data for {life_bin}', transform=ax.transAxes, ha='center', va='center')
            ax.set_axis_off()
            continue
        count_min = int(panel['n_samples'].min()) if panel['n_samples'].notna().any() else 0
        count_max = int(panel['n_samples'].max()) if panel['n_samples'].notna().any() else 0
        ax.set_title(f'{life_bin}{chr(10)}(n={count_min}-{count_max})')
        ax.set_xlabel('Training fraction (%)')
        ax.set_ylabel(f'{metric_label}(log-life) [decades]')
        ax.grid(True, alpha=0.3)
    handles, labels = axes[0].get_legend_handles_labels() if axes else ([], [])
    if handles:
        fig.legend(handles, labels, loc='upper center', ncol=min(4, len(labels)), fontsize=8)
    fig.suptitle(f'{metric_label} vs training fraction by life bin')
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    save_fig(fig, filename)



def full_set_summary_plot(filename: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
    for ax, metric_col, metric_label in [
        (axes[0], 'mae_loglife', 'MAE'),
        (axes[1], 'rmse_loglife', 'RMSE'),
    ]:
        subset = full_set_df.copy()
        eh.assert_non_empty_plot_df(
            subset.dropna(subset=[metric_col]) if metric_col in subset.columns else subset.iloc[0:0],
            intended_plot=f'Notebook 3 full-test-set {metric_label} plot',
            reference_df=life_bands,
            metric_cols=(metric_col,),
        )
        _plot_runs_and_summary(ax, subset, metric_col)
        ax.set_title(f'{eh.FULL_TEST_SET_LABEL}: {metric_label}')
        ax.set_xlabel('Training fraction (%)')
        ax.set_ylabel(f'{metric_label}(log-life) [decades]')
        ax.grid(True, alpha=0.3)
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper center', ncol=min(4, len(labels)), fontsize=8)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    save_fig(fig, filename)


faceted_metric_plot('mae_loglife', 'MAE', 'fraction_vs_life_bin_mae')
faceted_metric_plot('rmse_loglife', 'RMSE', 'fraction_vs_life_bin_rmse')
full_set_summary_plot('fraction_vs_full_test_set_mae_rmse')

display(summary_by_fraction)
display(life_bands[['model_family', 'training_fraction', 'checkpoint_path', 'training_config_id', 'life_bin', 'n_samples', 'mae_loglife', 'rmse_loglife']].head(20))




## Cautious interpretation


In [ ]:
def plateau_fraction(sub, metric, tol_rel=0.05, tol_abs=0.01):
    sub = sub.sort_values('training_fraction')
    full = sub[sub.training_fraction == 1.0]
    if full.empty or sub.empty:
        return None
    target = float(full.iloc[0][metric])
    tol = max(abs(target) * tol_rel, tol_abs)
    eligible = sub[sub[metric] <= target + tol]
    if eligible.empty:
        return None
    return float(eligible.iloc[0]['training_fraction'])

def first_fraction_meeting(sub, metric, threshold):
    sub = sub.sort_values('training_fraction')
    eligible = sub[sub[metric] <= threshold]
    return None if eligible.empty else float(eligible.iloc[0]['training_fraction'])

argent = summary_by_fraction[summary_by_fraction.model_family == 'ArGEnT_self_att_noSDF'].copy()
fp = summary_by_fraction[summary_by_fraction.model_family == 'PointNetMLPJoint_FP'].copy()
argent_plateau = plateau_fraction(argent, 'LogLife_MAE')
fp_plateau = plateau_fraction(fp, 'LogLife_MAE')
argent_full = float(argent[argent.training_fraction == 1.0]['LogLife_MAE'].iloc[0]) if not argent[argent.training_fraction == 1.0].empty else np.nan
fp_reaches_argent_full = first_fraction_meeting(fp, 'LogLife_MAE', argent_full) if not np.isnan(argent_full) else None
summary_lines = [
    '# Data-efficiency summary',
    '',
    f'Repository commit: `{COMMIT}`',
    '',
    'Evaluation label: **validation-split evaluation**. Every curve uses the same deterministic 20% geometry holdout (seed 42).',
    '',
    'Qualitative illustrative examples are generated only from local example HDF5 files; quantitative metrics here use production validation-split data only.',
    '',
    '## Plateau and efficiency read-out',
    '',
    f"- ArGEnT pooled-LogLife plateau fraction (within 5% or 0.01 absolute of full-data performance): {argent_plateau if argent_plateau is not None else 'not reached'}.",
    f"- PointNet+FP pooled-LogLife plateau fraction (same rule): {fp_plateau if fp_plateau is not None else 'not reached'}.",
    f"- Smallest FP fraction reaching ArGEnT full-data pooled LogLife MAE: {fp_reaches_argent_full if fp_reaches_argent_full is not None else 'not reached'}.",
    '',
    '## Interpretation',
    '',
    'Interpret pooled metrics together with full life-band, grouped-region, and geometry-level diagnostics.',
]
summary_text = "\\n".join(summary_lines)
(RESULTS_DIR / 'comparison_summary.md').write_text(summary_text, encoding='utf-8')
display(Markdown(summary_text))
